# Fine-Tuning BERT for POS Tagging & Chunking

**Objective:** Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.

---
**Pipeline:** Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

## Setup: Install Required Libraries

In [ ]:
# Install required packages
!pip install transformers datasets seqeval evaluate accelerate -q

In [ ]:
# Core imports
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

---
## Task 1: Dataset Selection (10%)

We use **CoNLL-2003** — a widely used benchmark for NLP sequence labeling tasks.

It provides **two annotation layers** in a single dataset:
- **POS Tags** — Part-of-Speech labels (grammar-level)
- **Chunk Tags** — Phrase/Chunk labels (phrase-level, IOB format)

This lets us train and compare both tasks without needing two separate datasets.

In [ ]:
# ─────────────────────────────────────────────
# TASK 1: Dataset Selection
# ─────────────────────────────────────────────

# Load CoNLL-2003 dataset
raw_dataset = load_dataset("conll2003", trust_remote_code=True)

print("Dataset structure:")
print(raw_dataset)
print()

# Inspect a sample
sample = raw_dataset["train"][0]
print("Sample tokens :", sample["tokens"])
print("Sample POS tags:", sample["pos_tags"])
print("Sample Chunks  :", sample["chunk_tags"])

In [ ]:
# Extract label lists from dataset features
pos_label_list   = raw_dataset["train"].features["pos_tags"].feature.names
chunk_label_list = raw_dataset["train"].features["chunk_tags"].feature.names

print(f"POS Tags ({len(pos_label_list)} labels):")
print(pos_label_list)
print()
print(f"Chunk Tags ({len(chunk_label_list)} labels):")
print(chunk_label_list)

In [ ]:
# ── Task 1 Deliverable ──────────────────────────────────────────────────────
print("="*60)
print("TASK 1 DELIVERABLES")
print("="*60)
print(f"Dataset      : CoNLL-2003")
print(f"Source       : Hugging Face Datasets Hub")
print(f"Train samples: {len(raw_dataset['train'])}")
print(f"Val   samples: {len(raw_dataset['validation'])}")
print(f"Test  samples: {len(raw_dataset['test'])}")
print()
print(f"Label Type 1 — POS Tags  ({len(pos_label_list)} labels)")
print(f"  Purpose: Grammar-level word tagging")
print(f"  Example: NN (noun), VBZ (verb), DT (determiner)")
print()
print(f"Label Type 2 — Chunk Tags ({len(chunk_label_list)} labels)")
print(f"  Purpose: Phrase-level grouping (IOB2 format)")
print(f"  Example: B-NP (begin noun phrase), I-VP (inside verb phrase)")
print("="*60)

---
## Task 2: Data Preprocessing (15%)

Key challenge: BERT uses **subword tokenization** (WordPiece), so one word may split into multiple tokens. We must **align labels** so that:
- The **first subword** of each word gets the real label
- All **continuation subwords** get `-100` (ignored by the loss function)
- Special tokens `[CLS]` and `[SEP]` also get `-100`

In [ ]:
# ─────────────────────────────────────────────
# TASK 2: Data Preprocessing
# ─────────────────────────────────────────────

MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# ── Subword alignment illustration ──────────────────────────────────────────
demo_sentence = ["EU", "rejects", "German", "call"]
demo_encoding = tokenizer(demo_sentence, is_split_into_words=True)

print("Subword Tokenization Demo")
print("-"*40)
print("Original tokens :", demo_sentence)
print("Subword tokens  :", tokenizer.convert_ids_to_tokens(demo_encoding["input_ids"]))
print("Word IDs        :", demo_encoding.word_ids())
print()
print("Note: None = special token ([CLS]/[SEP]), integers = word index")

In [ ]:
def tokenize_and_align_labels(examples, label_column="pos_tags"):
    """
    Tokenize words with BERT tokenizer and align labels to subword tokens.

    Strategy:
      - First subword of a word → real label
      - Continuation subwords  → -100 (ignored by cross-entropy loss)
      - Special tokens         → -100
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,   # input is already word-tokenized
        padding="max_length",
        max_length=128,
    )

    all_labels = []
    for i, label_ids in enumerate(examples[label_column]):
        word_ids    = tokenized_inputs.word_ids(batch_index=i)
        prev_word_id = None
        aligned_labels = []

        for word_id in word_ids:
            if word_id is None:
                # [CLS] or [SEP] → ignore
                aligned_labels.append(-100)
            elif word_id != prev_word_id:
                # First subword of a new word → assign real label
                aligned_labels.append(label_ids[word_id])
            else:
                # Continuation subword → ignore
                aligned_labels.append(-100)

            prev_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

print("Label alignment function defined.")

In [ ]:
# ── Build POS dataset ────────────────────────────────────────────────────────
pos_tokenized = raw_dataset.map(
    lambda x: tokenize_and_align_labels(x, label_column="pos_tags"),
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)

# ── Build Chunk dataset ──────────────────────────────────────────────────────
chunk_tokenized = raw_dataset.map(
    lambda x: tokenize_and_align_labels(x, label_column="chunk_tags"),
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)

print("Tokenized POS dataset:  ", pos_tokenized)
print("Tokenized Chunk dataset:", chunk_tokenized)

In [ ]:
# ── Task 2 Deliverable: Verify outputs ──────────────────────────────────────
sample_idx = 0
print("TASK 2 DELIVERABLES — Sample inspection (index 0)")
print("="*60)

sample_pos = pos_tokenized["train"][sample_idx]

print(f"input_ids      (first 20): {sample_pos['input_ids'][:20]}")
print(f"attention_mask (first 20): {sample_pos['attention_mask'][:20]}")
print(f"labels         (first 20): {sample_pos['labels'][:20]}")
print()
print("Token | Label  | Meaning")
print("-"*50)
tokens = tokenizer.convert_ids_to_tokens(sample_pos["input_ids"][:15])
labels = sample_pos["labels"][:15]
for tok, lbl in zip(tokens, labels):
    if lbl == -100:
        meaning = "(special / subword — ignored)"
        lbl_str = "-100"
    else:
        meaning = pos_label_list[lbl]
        lbl_str = str(lbl)
    print(f"  {tok:<15} {lbl_str:<6} {meaning}")

---
## Task 3: Model Setup (15%)

We load `distilbert-base-uncased` with a token classification head.
The head is a linear layer projecting BERT's hidden states → `num_labels` classes.

In [ ]:
# ─────────────────────────────────────────────
# TASK 3: Model Setup
# ─────────────────────────────────────────────

def build_model(label_list, model_checkpoint=MODEL_CHECKPOINT):
    """Build AutoModelForTokenClassification with correct label mappings."""
    id2label = {i: lbl for i, lbl in enumerate(label_list)}
    label2id = {lbl: i for i, lbl in enumerate(label_list)}

    model = AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    return model, id2label, label2id


# Build POS model
pos_model, pos_id2label, pos_label2id = build_model(pos_label_list)

# Build Chunk model
chunk_model, chunk_id2label, chunk_label2id = build_model(chunk_label_list)

print("POS Model")
print(f"  num_labels : {pos_model.config.num_labels}")
print(f"  id2label   : {pos_id2label}")
print()
print("Chunk Model")
print(f"  num_labels : {chunk_model.config.num_labels}")
print(f"  id2label   : {chunk_id2label}")

In [ ]:
# Count trainable parameters
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_params(pos_model)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Architecture        : {MODEL_CHECKPOINT}")
print(f"Task head           : AutoModelForTokenClassification")

---
## Task 4: Training (20%)

We use Hugging Face's `Trainer` API with the following configuration:

| Hyperparameter | Value |
|---|---|
| Learning rate | 2e-5 |
| Epochs | 3 |
| Batch size | 16 |
| Weight decay | 0.01 |
| Evaluation | Per epoch |

In [ ]:
# ─────────────────────────────────────────────
# TASK 4: Training
# ─────────────────────────────────────────────

seqeval = evaluate.load("seqeval")

def make_compute_metrics(id2label):
    """Returns a compute_metrics function for seqeval evaluation."""
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions = np.argmax(logits, axis=-1)

        true_labels = [
            [id2label[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]
        results = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            "precision": results["overall_precision"],
            "recall"   : results["overall_recall"],
            "f1"       : results["overall_f1"],
            "accuracy" : results["overall_accuracy"],
        }
    return compute_metrics


def get_training_args(output_dir, epochs=3, batch_size=16, lr=2e-5):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_dir=f"{output_dir}/logs",
        logging_steps=50,
        report_to="none",           # disable wandb / external loggers
        fp16=torch.cuda.is_available(),
    )

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Training utilities defined.")

In [ ]:
# ── Train POS Model ──────────────────────────────────────────────────────────
print("Training POS Tagging Model...")
print("="*50)

pos_trainer = Trainer(
    model=pos_model,
    args=get_training_args("./pos_model"),
    train_dataset=pos_tokenized["train"],
    eval_dataset=pos_tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_id2label),
)

pos_train_result = pos_trainer.train()
print("\nPOS Training complete!")
print(pos_train_result.metrics)

In [ ]:
# ── Train Chunk Model ────────────────────────────────────────────────────────
print("Training Chunking Model...")
print("="*50)

chunk_trainer = Trainer(
    model=chunk_model,
    args=get_training_args("./chunk_model"),
    train_dataset=chunk_tokenized["train"],
    eval_dataset=chunk_tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_id2label),
)

chunk_train_result = chunk_trainer.train()
print("\nChunking Training complete!")
print(chunk_train_result.metrics)

---
## Task 5: Evaluation (15%)

We evaluate both models on the **held-out test set** using `seqeval` which computes sequence-level precision, recall, and F1 — stricter than token-level accuracy.

In [ ]:
# ─────────────────────────────────────────────
# TASK 5: Evaluation
# ─────────────────────────────────────────────

pos_eval   = pos_trainer.evaluate(pos_tokenized["test"])
chunk_eval = chunk_trainer.evaluate(chunk_tokenized["test"])

print("="*60)
print("TASK 5 — EVALUATION RESULTS (Test Set)")
print("="*60)

print("\n[POS Tagging Model]")
print(f"  Precision : {pos_eval['eval_precision']:.4f}")
print(f"  Recall    : {pos_eval['eval_recall']:.4f}")
print(f"  F1 Score  : {pos_eval['eval_f1']:.4f}")
print(f"  Accuracy  : {pos_eval['eval_accuracy']:.4f}")

print("\n[Chunking Model]")
print(f"  Precision : {chunk_eval['eval_precision']:.4f}")
print(f"  Recall    : {chunk_eval['eval_recall']:.4f}")
print(f"  F1 Score  : {chunk_eval['eval_f1']:.4f}")
print(f"  Accuracy  : {chunk_eval['eval_accuracy']:.4f}")

In [ ]:
# ── Per-class analysis ────────────────────────────────────────────────────────
def detailed_eval(trainer, tokenized_data, id2label, task_name):
    """Run full seqeval evaluation with per-class breakdown."""
    predictions_output = trainer.predict(tokenized_data["test"])
    preds  = np.argmax(predictions_output.predictions, axis=-1)
    labels = predictions_output.label_ids

    true_labels = [[id2label[l] for l in row if l != -100] for row in labels]
    true_preds  = [[id2label[p] for p, l in zip(pr, lr) if l != -100]
                   for pr, lr in zip(preds, labels)]

    results = seqeval.compute(
        predictions=true_preds,
        references=true_labels,
    )

    print(f"\n{'='*60}")
    print(f"Per-Class Results — {task_name}")
    print(f"{'='*60}")
    print(f"{'Label':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-"*55)
    for label, scores in sorted(results.items()):
        if isinstance(scores, dict):
            print(f"{label:<12} {scores['precision']:>10.4f} "
                  f"{scores['recall']:>10.4f} {scores['f1-score']:>10.4f} "
                  f"{scores['number']:>10}")
    print("-"*55)
    print(f"{'Overall':<12} {results['overall_precision']:>10.4f} "
          f"{results['overall_recall']:>10.4f} "
          f"{results['overall_f1']:>10.4f}")
    return results

pos_detailed   = detailed_eval(pos_trainer,   pos_tokenized,   pos_id2label,   "POS Tagging")
chunk_detailed = detailed_eval(chunk_trainer, chunk_tokenized, chunk_id2label, "Chunking")

---
## Task 6: Inference (10%)

Load the trained models and run prediction on custom sentences.

In [ ]:
# ─────────────────────────────────────────────
# TASK 6: Inference
# ─────────────────────────────────────────────

# Save models
pos_trainer.save_model("./pos_model_final")
chunk_trainer.save_model("./chunk_model_final")
tokenizer.save_pretrained("./pos_model_final")
tokenizer.save_pretrained("./chunk_model_final")
print("Models saved.")

In [ ]:
# ── Inference helper ─────────────────────────────────────────────────────────
def predict_tags(sentence, model, tokenizer, id2label):
    """
    Predict token-level tags for a raw sentence string.
    Returns list of (word, predicted_tag) tuples.
    """
    words = sentence.split()
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )
    word_ids = encoding.word_ids()

    model.eval()
    with torch.no_grad():
        outputs = model(**{k: v.to(model.device) for k, v in encoding.items()})

    logits      = outputs.logits[0].cpu()
    predictions = torch.argmax(logits, dim=-1).numpy()

    # Map back to word-level (first subword only)
    results = []
    prev_word_id = None
    for idx, word_id in enumerate(word_ids):
        if word_id is None or word_id == prev_word_id:
            pass
        else:
            results.append((words[word_id], id2label[predictions[idx]]))
        prev_word_id = word_id

    return results

print("Inference function defined.")

In [ ]:
# Load saved models for inference
loaded_pos_model = AutoModelForTokenClassification.from_pretrained("./pos_model_final")
loaded_chunk_model = AutoModelForTokenClassification.from_pretrained("./chunk_model_final")
inference_tokenizer = AutoTokenizer.from_pretrained("./pos_model_final")

# ── Test sentences ───────────────────────────────────────────────────────────
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "Apple announced a new iPhone at the annual conference",
]

for sentence in test_sentences:
    pos_tags   = predict_tags(sentence, loaded_pos_model,   inference_tokenizer, pos_id2label)
    chunk_tags = predict_tags(sentence, loaded_chunk_model, inference_tokenizer, chunk_id2label)

    print(f"\nInput: {sentence}")
    print("─"*55)
    print(f"{'Word':<18} {'POS Tag':<12} {'Chunk Tag':<15}")
    print("─"*55)
    for (word, pos), (_, chunk) in zip(pos_tags, chunk_tags):
        print(f"{word:<18} {pos:<12} {chunk:<15}")
    print()

---
## Task 7: Comparison — POS Tagging vs Chunking (10%)

In [ ]:
# ─────────────────────────────────────────────
# TASK 7: Comparison
# ─────────────────────────────────────────────

comparison_table = {
    "Aspect": [
        "Task Type",
        "Granularity",
        "Label Scope",
        "Label Count (CoNLL-2003)",
        "Label Format",
        "Example Output",
        "Difficulty",
        "Use Case",
        "Test F1 (this run)",
    ],
    "POS Tagging": [
        "Grammar-level word classification",
        "Single token",
        "Per word",
        f"{len(pos_label_list)} tags",
        "Flat labels (NN, VBZ, DT …)",
        "John/NNP works/VBZ at/IN Google/NNP",
        "Easier — independent per token",
        "Syntax parsing, MT, QA",
        f"{pos_eval['eval_f1']:.4f}",
    ],
    "Chunking": [
        "Phrase-level span grouping",
        "Multi-token spans",
        "Per phrase group",
        f"{len(chunk_label_list)} tags",
        "IOB2 (B-NP, I-VP, O …)",
        "[John] B-NP [works] B-VP [at Google] B-PP",
        "Harder — requires span consistency",
        "IE, NER, question answering",
        f"{chunk_eval['eval_f1']:.4f}",
    ],
}

print("\nTASK 7 — POS TAGGING vs CHUNKING COMPARISON")
print("="*85)
print(f"{'Aspect':<28} {'POS Tagging':<28} {'Chunking':<28}")
print("─"*85)
for i, aspect in enumerate(comparison_table["Aspect"]):
    pos_val   = comparison_table["POS Tagging"][i]
    chunk_val = comparison_table["Chunking"][i]
    print(f"{aspect:<28} {pos_val:<28} {chunk_val:<28}")
print("="*85)

In [ ]:
# ── Visual comparison bar chart ──────────────────────────────────────────────
import matplotlib.pyplot as plt

metrics   = ["Precision", "Recall", "F1 Score", "Accuracy"]
pos_vals  = [pos_eval["eval_precision"],  pos_eval["eval_recall"],
             pos_eval["eval_f1"],         pos_eval["eval_accuracy"]]
chunk_vals= [chunk_eval["eval_precision"],chunk_eval["eval_recall"],
             chunk_eval["eval_f1"],       chunk_eval["eval_accuracy"]]

x = np.arange(len(metrics))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, pos_vals,   w, label="POS Tagging", color="steelblue")
bars2 = ax.bar(x + w/2, chunk_vals, w, label="Chunking",    color="darkorange")

ax.set_xlabel("Metric",     fontsize=12)
ax.set_ylabel("Score",      fontsize=12)
ax.set_title("POS Tagging vs Chunking — Test Set Performance", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.legend()

for bar in bars1 + bars2:
    ax.annotate(f"{bar.get_height():.3f}",
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("comparison_chart.png", dpi=150)
plt.show()
print("Chart saved as comparison_chart.png")

---
## Task 8: Report / Blog (5%)

### POS Tagging vs Chunking with BERT — Observations & Insights

---

#### 1. What is POS Tagging?

Part-of-Speech (POS) tagging assigns a grammatical category to every word in a sentence. Each token receives exactly one flat label — for example: `John/NNP`, `runs/VBZ`, `fast/RB`. Labels encode surface-level grammar: whether a word is a noun, verb, adjective, adverb, preposition, and so on.

POS tagging is a **single-token, context-free classification task** — the label for a token doesn't constrain what labels adjacent tokens can have. This makes it relatively straightforward for a transformer model; even a small model achieves very high F1 on CoNLL-2003.

---

#### 2. What is Chunking?

Chunking (shallow parsing) groups adjacent words into **non-overlapping phrases** — noun phrases (NP), verb phrases (VP), prepositional phrases (PP), etc. Labels follow the **IOB2 format**:

- `B-NP` — beginning of a noun phrase  
- `I-NP` — inside (continuation of) a noun phrase  
- `O`    — outside any phrase  

Chunking is **span-level**, meaning the model must not only predict a label for each token but also ensure that `I-*` labels never appear without a preceding `B-*` of the same type. This span coherence requirement makes chunking inherently harder.

---

#### 3. Key Differences

| Dimension | POS Tagging | Chunking |
|---|---|---|
| Output | Per-token flat label | Per-token IOB label |
| Dependency | Labels are independent | Labels must form valid spans |
| Label space | ~45 Penn Treebank tags | ~23 IOB chunk tags |
| Difficulty | Easier | Harder |
| Linguistic level | Morpho-syntactic | Syntactic |

---

#### 4. Challenges Faced

**Subword Tokenization Alignment**  
BERT's WordPiece tokenizer splits words like *"running"* into `["run", "##ning"]`. Since our original labels are word-level, we had to implement a careful alignment strategy: assign the real label to the first subword and `-100` to all continuations. This `-100` sentinel tells PyTorch's cross-entropy loss to ignore those positions.

**IOB Consistency in Chunking**  
The model can occasionally predict sequences like `I-NP` after `O` — a malformed span. One mitigation is to post-process predictions and convert invalid `I-*` beginnings into `B-*`. More robust approaches (CRF layer, constrained decoding) exist but were beyond this assignment's scope.

**Class Imbalance**  
In chunking, the `O` (outside) tag dominates the dataset — roughly 15–20 % of tokens. This can inflate accuracy while individual phrase types remain harder to detect. `seqeval` mitigates this by computing entity/chunk-level F1, not token-level accuracy.

**Compute Constraints**  
We chose DistilBERT (66 M parameters) over full BERT-base (110 M) for faster iteration. DistilBERT retains ~97 % of BERT's performance at 60 % of the size, making it suitable for prototyping on limited hardware.

---

#### 5. Observations & Insights

- **POS tagging is near-saturated.** BERT achieves > 98 % accuracy on Penn Treebank. CoNLL-2003 POS accuracy is similarly very high. The remaining errors are almost always genuine ambiguities (e.g., "He can can the can" — three different uses of the same word form).

- **Chunking benefits enormously from BERT's contextual embeddings.** Earlier systems (HMMs, CRFs with hand-crafted features) struggled with long-range dependencies inside phrases. BERT's self-attention captures multi-word coherence naturally.

- **seqeval is stricter than token accuracy.** A phrase is counted as correctly identified only if *every token* in the span has the right label. This is a realistic measure for downstream NLP use.

- **Fine-tuning is data-efficient.** Only 3 epochs were needed; the pre-trained weights already encode most morpho-syntactic knowledge. Additional epochs risk overfitting on the relatively small CoNLL-2003 training set (~14 K sentences).

- **Error analysis matters.** Comparing per-class F1 reveals that rare tags (e.g., `CONJP` conjunctive phrases, `UCP` unlike coordinated phrases) have lower scores due to fewer training examples — a direction for future data augmentation.

---

#### 6. Conclusion

Fine-tuning DistilBERT for both POS tagging and chunking demonstrates the versatility of pre-trained language models for sequence labeling. The same architecture, same tokenizer, and same training loop serve both tasks; only the label space and dataset change. POS tagging is simpler and achieves near-perfect scores; chunking is richer linguistically and more challenging, but still attains strong F1 with minimal task-specific engineering. Together, these two tasks form the backbone of traditional NLP pipelines and remain important building blocks for more complex systems like dependency parsing, named entity recognition, and information extraction.

In [ ]:
# ── Final Summary Cell ───────────────────────────────────────────────────────
print("\n" + "="*65)
print("  ASSIGNMENT SUMMARY")
print("="*65)
print(f"  Model checkpoint : {MODEL_CHECKPOINT}")
print(f"  Dataset          : CoNLL-2003")
print()
print("  Task                  | F1 Score   | Precision  | Recall")
print("  " + "-"*58)
print(f"  POS Tagging           | {pos_eval['eval_f1']:.4f}     | "
      f"{pos_eval['eval_precision']:.4f}     | {pos_eval['eval_recall']:.4f}")
print(f"  Chunking              | {chunk_eval['eval_f1']:.4f}     | "
      f"{chunk_eval['eval_precision']:.4f}     | {chunk_eval['eval_recall']:.4f}")
print("="*65)
print("  All 8 tasks completed successfully!")
print("="*65)